In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

# Setup synthetic regression data for Tasks 1 & 2
X_reg, y_reg = make_regression(n_samples=200, n_features=10, n_informative=5, random_state=42)
feature_names = [f"Feature_{i}" for i in range(10)]
df_reg = pd.DataFrame(X_reg, columns=feature_names)

In [2]:
# Initialize base estimator
estimator = LinearRegression()

# Initialize RFE to select top 5 features
rfe = RFE(estimator=estimator, n_features_to_select=5)
rfe.fit(X_reg, y_reg)

# Summarize chosen features
selected_features = [feature_names[i] for i in range(len(feature_names)) if rfe.support_[i]]
print("Task 1 - Selected Top 5 Features:", selected_features)

Task 1 - Selected Top 5 Features: ['Feature_0', 'Feature_2', 'Feature_3', 'Feature_4', 'Feature_9']


In [3]:
# Take a subset of 2 features to easily visualize the transformation
sample_data = df_reg[['Feature_0', 'Feature_1']]

# Initialize PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(sample_data)

# Convert to DataFrame to see the generated feature names
poly_cols = poly.get_feature_names_out(['Feature_0', 'Feature_1'])
df_poly = pd.DataFrame(poly_features, columns=poly_cols)

print("\nTask 2 - Original shape:", sample_data.shape)
print("Task 2 - Polynomial shape (degree=2):", df_poly.shape)
print("Generated columns:", list(df_poly.columns))


Task 2 - Original shape: (200, 2)
Task 2 - Polynomial shape (degree=2): (200, 5)
Generated columns: ['Feature_0', 'Feature_1', 'Feature_0^2', 'Feature_0 Feature_1', 'Feature_1^2']


In [4]:
# Create a mock dataset with mixed data types
data = {
    'Age': [25, np.nan, 47, 35, 22, 58],
    'Fare': [7.25, 71.28, 53.10, 8.05, 8.45, 51.86],
    'Embarked': ['S', 'C', 'S', 'S', np.nan, 'Q'],
    'Sex': ['male', 'female', 'female', 'male', 'male', 'female'],
    'Survived': [0, 1, 1, 0, 0, 1]
}
df_mixed = pd.DataFrame(data)

X_mixed = df_mixed.drop('Survived', axis=1)
y_mixed = df_mixed['Survived']

# Define separate pipelines for numerical and categorical features
numeric_features = ['Age', 'Fare']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_features = ['Embarked', 'Sex']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Create the master pipeline including the classifier
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression())
])

# Fit the entire chain
full_pipeline.fit(X_mixed, y_mixed)
print("\nTasks 3 & 4 - Mixed type pipeline successfully trained!")


Tasks 3 & 4 - Mixed type pipeline successfully trained!


In [5]:
# Synthetic Titanic Dataset representing engineered structural challenges
np.random.seed(42)
n_rows = 400
titanic_data = {
    'Age': np.random.choice([22, 38, 26, np.nan, 35, 54, 2], size=n_rows),
    'Fare': np.random.uniform(5, 150, size=n_rows),
    'SibSp': np.random.randint(0, 4, size=n_rows),
    'Parch': np.random.randint(0, 3, size=n_rows),
    'Sex': np.random.choice(['male', 'female'], size=n_rows),
    'Pclass': np.random.choice([1, 2, 3], size=n_rows)
}
df_titanic = pd.DataFrame(titanic_data)
# Simple generation rule for targets
y_titanic = np.where((df_titanic['Sex'] == 'female') | (df_titanic['Fare'] > 50), 1, 0)

# --- Feature Engineering Challenge ---
# 1. FamilySize = SibSp + Parch + 1
df_titanic['FamilySize'] = df_titanic['SibSp'] + df_titanic['Parch'] + 1

# 2. IsAlone = 1 if FamilySize == 1 else 0
df_titanic['IsAlone'] = np.where(df_titanic['FamilySize'] == 1, 1, 0)

# 3. Age_Class_Interaction = Age * Pclass (Handle missing values first for safety)
df_titanic['Age_Filled'] = df_titanic['Age'].fillna(df_titanic['Age'].median())
df_titanic['Age_Class'] = df_titanic['Age_Filled'] * df_titanic['Pclass']
df_titanic.drop('Age_Filled', axis=1, inplace=True)

# Split features and labels
X_t = df_titanic.drop(['SibSp', 'Parch'], axis=1) # Dropping components used for engineering

# Set up preprocessor groups
num_cols = ['Age', 'Fare', 'FamilySize', 'IsAlone', 'Age_Class']
cat_cols = ['Sex', 'Pclass'] # Treating Pclass as categorical

titanic_preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols),
    ('cat', Pipeline([('enc', OneHotEncoder(drop='first'))]), cat_cols)
])

# Final end-to-end pipeline
titanic_pipeline = Pipeline([
    ('preprocess', titanic_preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_t, y_titanic, test_size=0.2, random_state=42)
titanic_pipeline.fit(X_train_t, y_train_t)

print("\n--- Practice Sheet Result ---")
print(f"Titanic End-to-End Accuracy: {titanic_pipeline.score(X_test_t, y_test_t):.4f}")


--- Practice Sheet Result ---
Titanic End-to-End Accuracy: 0.9875
